# Optimized Dental Caries Detection - YOLOv8x Segmentation
## Target: mAP@50 > 70% | High Precision & Recall
### CLAHE Preprocessing + 800x800 Resolution + Advanced Training

In [1]:
!nvidia-smi


Wed Mar 11 04:12:56 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.161.08             Driver Version: 535.161.08   CUDA Version: 13.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  | 00000000:07:00.0 Off |                   On |
| N/A   26C    P0              56W / 400W |                  N/A |     N/A      Default |
|                                         |                      |              Enabled |
+-----------------------------------------+----------------------+--

## Phase 1: Install Dependencies

In [19]:
!pip install -q ultralytics matplotlib pillow numpy torch torchvision
!pip uninstall -y opencv-python opencv-python-headless

# 2. Install the server-safe, GUI-free version
!pip install opencv-python-headless

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (60.4 MB)


## Phase 2: Imports & Setup

In [20]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import json
import shutil
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)

Path('dataset_preprocessed/images/train').mkdir(parents=True, exist_ok=True)
Path('dataset_preprocessed/images/val').mkdir(parents=True, exist_ok=True)
Path('dataset_preprocessed/labels/train').mkdir(parents=True, exist_ok=True)
Path('dataset_preprocessed/labels/val').mkdir(parents=True, exist_ok=True)
Path('results').mkdir(exist_ok=True)

print("Setup complete!")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB MIG 1g.10gb
Setup complete!


## Phase 6: Copy Labels

In [21]:
print("Copying labels...")

for lbl in Path('train/labels').glob('*.txt'):
    shutil.copy(lbl, f'dataset_preprocessed/labels/train/{lbl.name}')

for lbl in Path('val/labels').glob('*.txt'):
    shutil.copy(lbl, f'dataset_preprocessed/labels/val/{lbl.name}')

print("Labels copied!")

Copying labels...
Labels copied!


## Phase 7: Create YAML Config

In [22]:
yaml_text = f"""path: {Path('dataset').absolute()}
train: images/train
val: images/val

nc: 1
names:
  0: caries
"""

with open('data.yaml', 'w') as f:
    f.write(yaml_text)

print("YAML config created!")
print(f"Train: {train_count} images")
print(f"Val: {val_count} images")

YAML config created!
Train: 560 images
Val: 70 images


## Phase 8: Load YOLOv8l-seg Model

In [23]:
print("Loading YOLOv8m (segmentation) (med)...")
model = YOLO('yolov8m-seg.pt')
print("Model loaded!")

Loading YOLOv8m (segmentation) (med)...
Model loaded!


## Phase 9: Train with Optimized Settings

In [24]:
print("Starting training...")
print("Key optimizations:")
print("  - Image size: 800x800 (was 640x640)")
print("  - Model: YOLOv8x-seg (was YOLOv8m-seg)")
print("  - Epochs: 150 (was 50)")
print("  - Optimizer: AdamW (was SGD)")
print("  - Augmentation: Mosaic + Copy-Paste")
print("  - Preprocessing: CLAHE applied")
print()

results = model.train(
    data='data.yaml',
    epochs=150,
    patience=50,
    batch=2,
    imgsz=800,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.3,
    label_smoothing=0.1,
    overlap_mask=True,
    mask_ratio=4,
    device=device,
    workers=8,
    project='runs',
    name='caries_optimized',
    exist_ok=True,
    pretrained=True,
    verbose=True,
    seed=42,
    single_cls=True,
    amp=True,
    val=True,
    plots=True,
    save=True
)

print("Training complete!")

Starting training...
Key optimizations:
  - Image size: 800x800 (was 640x640)
  - Model: YOLOv8x-seg (was YOLOv8m-seg)
  - Epochs: 150 (was 50)
  - Optimizer: AdamW (was SGD)
  - Augmentation: Mosaic + Copy-Paste
  - Preprocessing: CLAHE applied

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.21 🚀 Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB MIG 1g.10gb, 9728MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, i

KeyboardInterrupt: 

## Phase 10: Validate with TTA

In [ ]:
print("Loading best model...")
best_model = YOLO('runs/caries_optimized/weights/best.pt')

print("Validating WITHOUT TTA...")
metrics_no_tta = best_model.val(
    data='data.yaml',
    imgsz=800,
    batch=8,
    conf=0.15,
    iou=0.4,
    max_det=300,
    augment=False
)

print("Validating WITH TTA...")
metrics_tta = best_model.val(
    data='data.yaml',
    imgsz=800,
    batch=8,
    conf=0.15,
    iou=0.4,
    max_det=300,
    augment=True
)

print("\nResults:")
print(f"mAP@50 without TTA: {metrics_no_tta.seg.map50:.4f} ({metrics_no_tta.seg.map50*100:.2f}%)")
print(f"mAP@50 with TTA:    {metrics_tta.seg.map50:.4f} ({metrics_tta.seg.map50*100:.2f}%)")
print(f"TTA improvement:    {(metrics_tta.seg.map50-metrics_no_tta.seg.map50)*100:+.2f}%")

metrics = metrics_tta

## Phase 11: Final Report

In [ ]:
f1 = 2*(metrics.box.mp*metrics.box.mr)/(metrics.box.mp+metrics.box.mr) if (metrics.box.mp+metrics.box.mr)>0 else 0

print("="*70)
print("FINAL RESULTS")
print("="*70)
print(f"\nmAP@50 (seg):     {metrics.seg.map50:.4f} ({metrics.seg.map50*100:.2f}%)")
print(f"mAP@50-95 (seg):  {metrics.seg.map:.4f} ({metrics.seg.map*100:.2f}%)")
print(f"Precision:        {metrics.box.mp:.4f} ({metrics.box.mp*100:.2f}%)")
print(f"Recall:           {metrics.box.mr:.4f} ({metrics.box.mr*100:.2f}%)")
print(f"F1 Score:         {f1:.4f} ({f1*100:.2f}%)")

print("\n" + "="*70)
print("TARGET ACHIEVEMENT")
print("="*70)
print(f"mAP@50 target: >70%")
print(f"mAP@50 achieved: {metrics.seg.map50*100:.2f}%")

if metrics.seg.map50 > 0.70:
    print("\nSTATUS: TARGET ACHIEVED!")
else:
    print("\nSTATUS: Needs improvement")
    print("Next steps:")
    print("  - Try YOLOv11x-seg")
    print("  - Increase to 1024px")
    print("  - Train 200+ epochs")
    print("  - Model ensemble")

print("\n" + "="*70)

## Phase 12: Inference Function

In [ ]:
def detect_caries(img_path, conf=0.15, use_tta=True, preprocess=True):
    if preprocess:
        temp = Path('temp.jpg')
        apply_clahe(img_path, temp)
        img_path = temp
    
    results = best_model.predict(
        source=str(img_path),
        conf=conf,
        iou=0.4,
        max_det=300,
        augment=use_tta,
        device=device,
        verbose=False
    )
    
    if preprocess and Path('temp.jpg').exists():
        Path('temp.jpg').unlink()
    
    return results[0]

print("Inference function ready!")
print("Usage: result = detect_caries('path/to/xray.jpg')")